# Experiment 10 — Rapid evidence audit and lower-confidence failure analysis

**Goal:** Answer the guide's remaining offline question in one run: do confidence + cross-view agreement selections preserve correct detections, especially in the **0.70–0.80** confidence band? Explain *which boxes are exchanged* at equal per-image K, and whether results differ by class.

**Run:** Kernel → Restart & Run All. Requires `numpy`, `pandas`, `matplotlib`, and the **existing** Experiment 10 outputs and validated COCO ground truth. **No GPU, Detectron2, or new inference required.** Does not modify the existing experiments. Saves CSV, PNG and Markdown into `Adaptive_teacher_Experiment/evidence_data/experiment_10_failure_audit/`.

**Scientific limits:** These are fixed-count pseudo-label diagnostic precision/recall/F1 (IoU ≥ 0.5), **not mAP**. GT is read only after loading already-selected predictions. Crowd annotations excluded without COCO crowd-ignore matching. No downstream training claim. Confidence band refers to **View-A proposal scores**, not teacher calibration.


In [ ]:
from pathlib import Path
import json, hashlib, datetime, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.home() / 'Desktop/phd/vizuara/Deliverables/milestone2'
EXP = ROOT / 'Adaptive_teacher_Experiment 2'
REPO = ROOT / 'Adaptive_teacher_Experiment'
RESULTS = EXP / 'experiment_10_weak_augmentation_agreement/results'
RANDOM = RESULTS / 'independent_weak_views_20'
GT = EXP / 'research_code_results/experiments/cityscapes_to_foggy/experiment_05_fog_aware_teacher/fog_aware_05c/evaluation_5000_v2/strong/exp05c_final_strong_val_coco_format.json'
OUT = REPO / 'evidence_data/experiment_10_failure_audit'
OUT.mkdir(parents=True, exist_ok=True)
FILES = {
    'flip50': RESULTS / 'matched_count_selections.json',
    'photo20': RANDOM / 'matched_count_selections.json',
    'gt': GT,
}
missing = [f'{k}: {p}' for k,p in FILES.items() if not p.is_file()]
if missing:
    raise FileNotFoundError('Required existing experiment files missing:\n' + '\n'.join(missing))
print('Found existing predictions and GT. Output:', OUT)
for k,p in FILES.items(): print(k, p)


## 1. Load fixed-count selections and audit ground truth
Selections were made in the prior frozen-teacher experiment **without consulting GT**. This notebook does not reselect predictions based on annotations.

In [ ]:
from collections import defaultdict
flip50 = json.loads(FILES['flip50'].read_text())
photo20 = json.loads(FILES['photo20'].read_text())
gt = json.loads(FILES['gt'].read_text())
ids = {Path(im['file_name']).name: im['id'] for im in gt['images']}
class_names = {int(c['id']):c['name'] for c in gt['categories']}
gt_by_id = defaultdict(list)
for a in gt['annotations']:
    if int(a.get('iscrowd',0)) != 0: continue
    x,y,w,h = a['bbox']
    gt_by_id[a['image_id']].append({'box':[x,y,x+w,y+h], 'class_id':int(a['category_id'])})
assert len(flip50)==50, f'Expected 50 flip scenes; found {len(flip50)}'
assert len(photo20)==20, f'Expected 20 photometric scenes; found {len(photo20)}'
assert len(gt['annotations'])==10412
assert sum(int(a.get('iscrowd',0))==0 for a in gt['annotations'])==10117
assert sum(int(a.get('iscrowd',0))!=0 for a in gt['annotations'])==295
for exp,records in [('flip50',flip50),('photo20',photo20)]:
    for r in records:
        assert Path(r['image']).name in ids, (exp,r['image'])
        assert len(r['confidence_only'])==len(r['confidence_agreement'])==int(r['k'])
print('PASS: 50 flip and 20 photometric scenes; per-image K equal for both methods')
print('PASS: 10,412 annotations = 10,117 non-crowd + 295 crowd')
print('Classes:',class_names)


## 2. Recompute class-aware one-to-one GT matching
Use the same greedy confidence-descending IoU≥0.5 protocol as the prior pilot. A proposal can be a TP in one selected set and an FP in another because matching is **set-dependent**; this is why we also report *selection swaps* separately.

In [ ]:
def iou(a,b):
    x1=max(float(a[0]),float(b[0])); y1=max(float(a[1]),float(b[1]))
    x2=min(float(a[2]),float(b[2])); y2=min(float(a[3]),float(b[3]))
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))
    bb=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))
    return inter/(aa+bb-inter) if aa+bb-inter>0 else 0.0

def match_selected(preds, targets):
    used=set(); results=[False]*len(preds)
    for pi in sorted(range(len(preds)),key=lambda i:-float(preds[i]['score'])):
        p=preds[pi]
        eligible=[(iou(p['box'],t['box']),j) for j,t in enumerate(targets)
                  if j not in used and int(p['class_id'])==int(t['class_id'])]
        if eligible:
            v,j=max(eligible)
            if v>=0.5: used.add(j);results[pi]=True
    return results

def signature(p):
    # Stable identity of a View-A proposal across two selection lists
    return (int(p['class_id']),round(float(p['score']),8),
            tuple(round(float(v),5) for v in p['box']))

image_rows=[]; box_rows=[]; swap_rows=[]
for experiment, records in [('original_flip_50',flip50),('independent_photometric_20',photo20)]:
    for r in records:
        fname=Path(r['image']).name
        targets=gt_by_id[ids[fname]]
        by_method={}
        for method in ['confidence_only','confidence_agreement']:
            selected=r[method]
            flags=match_selected(selected,targets)
            tp=sum(flags); k=len(selected)
            by_method[method]={signature(p):p for p in selected}
            image_rows.append({'experiment':experiment,'image':fname,'method':method,
                               'retained':k,'gt_objects':len(targets),'TP':tp,'FP':k-tp,'FN':len(targets)-tp})
            for p,flag in zip(selected,flags):
                score=float(p['score'])
                box_rows.append({'experiment':experiment,'image':fname,'method':method,
                                 'class_id':int(p['class_id']), 'class_name':class_names.get(int(p['class_id']),'unknown'),
                                 'score':score,'band':'0.70–0.80' if 0.70<=score<0.80 else ('<0.70' if score<0.70 else '≥0.80'),
                                 'TP':int(flag),'FP':int(not flag),
                                 'agrees':p.get('agrees',None),
                                 'agreement_iou':p.get('agreement_iou',None),
                                 'combined_score':p.get('combined_score',None)})
        a=set(by_method['confidence_only']); b=set(by_method['confidence_agreement'])
        swap_rows.append({'experiment':experiment,'image':fname,'K':int(r['k']),
                          'shared':len(a&b),'confidence_only_exclusive':len(a-b),
                          'agreement_exclusive':len(b-a),'swapped_pairs':len(a-b)})
image_df=pd.DataFrame(image_rows); box_df=pd.DataFrame(box_rows); swap_df=pd.DataFrame(swap_rows)
for name,df in [('per_image',image_df),('selected_boxes',box_df),('selection_swaps',swap_df)]:
    df.to_csv(OUT/f'{name}.csv',index=False)
print('PASS: matched-count GT diagnostics recomputed; CSV files saved')
print('Selection swaps:')
display(swap_df.groupby('experiment')[['K','shared','swapped_pairs']].sum())


## 3. Aggregate comparison and matched 20-scene control
The 20-scene flip and photometric setups use different candidate pools. Compare methods **within** a setup; do not attribute between-setup metric differences to augmentation alone.

In [ ]:
flip20_names={Path(r['image']).name for r in flip50[:20]}
flip20=image_df[(image_df.experiment=='original_flip_50') & image_df.image.isin(flip20_names)].copy()
flip20['experiment']='original_flip_20'
all_images=pd.concat([image_df,flip20],ignore_index=True)
agg=all_images.groupby(['experiment','method'],as_index=False)[['retained','gt_objects','TP','FP','FN']].sum()
agg['precision']=agg['TP']/agg['retained'].replace(0,np.nan)
agg['recall']=agg['TP']/agg['gt_objects'].replace(0,np.nan)
agg['F1']=2*agg.precision*agg.recall/(agg.precision+agg.recall).replace(0,np.nan)
agg.to_csv(OUT/'method_comparison.csv',index=False)
display(agg)
print('NOTE: F1 and recall use the existing diagnostic matching protocol; not COCO AP.')


## 4. Critical 0.70–0.80 confidence-band audit
Counts and GT-match rates of **selected** low-confidence boxes. Because these are selected subsets, these rates **do not estimate the accuracy of every 0.70–0.80 candidate**. For a complete candidate-pool analysis, raw `comparison` predictions must be matched separately.

In [ ]:
band=box_df.groupby(['experiment','method','band'],as_index=False).agg(
    selected=('TP','size'),TP=('TP','sum'),FP=('FP','sum'))
band['selected_precision']=band.TP/band.selected
band.to_csv(OUT/'selected_confidence_bands.csv',index=False)
display(band.sort_values(['experiment','band','method']))
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for ax,exp in zip(axes,['original_flip_50','independent_photometric_20']):
    sub=band[(band.experiment==exp)&(band.band=='0.70–0.80')]
    ax.bar(sub.method.replace({'confidence_only':'Confidence','confidence_agreement':'Confidence + agreement'}),
           sub.selected_precision*100)
    ax.set_ylim(0,100);ax.set_ylabel('GT-matched precision (%)');ax.set_title(exp+' — selected 0.70–0.80')
    for i,(_,r) in enumerate(sub.iterrows()):ax.text(i,float(r.selected_precision)*100+1,f'n={int(r.selected)}',ha='center',fontsize=9)
plt.tight_layout();plt.savefig(OUT/'low_confidence_selected_precision.png',dpi=170);plt.show()


## 5. Which classes and scenes lost true positives?

In [ ]:
class_table=box_df.groupby(['experiment','method','class_name'],as_index=False).agg(
    selected=('TP','size'),TP=('TP','sum'),FP=('FP','sum'))
class_table['precision']=class_table.TP/class_table.selected
class_table.to_csv(OUT/'classwise_selected_precision.csv',index=False)
display(class_table[class_table.experiment=='original_flip_50'].sort_values(['class_name','method']))
piv=image_df.pivot_table(index=['experiment','image'],columns='method',values='TP',aggfunc='first').reset_index()
piv['delta_TP_agreement_minus_confidence']=piv.confidence_agreement-piv.confidence_only
piv=piv.merge(swap_df[['experiment','image','swapped_pairs']],on=['experiment','image'])
piv.to_csv(OUT/'per_image_tp_deltas.csv',index=False)
print('Most harmed flip scenes (negative means agreement loses TPs):')
display(piv[piv.experiment=='original_flip_50'].sort_values('delta_TP_agreement_minus_confidence').head(12))
fig,ax=plt.subplots(figsize=(9,4.5))
for exp,g in piv.groupby('experiment'):
    ax.hist(g.delta_TP_agreement_minus_confidence,bins=np.arange(g.delta_TP_agreement_minus_confidence.min()-0.5,g.delta_TP_agreement_minus_confidence.max()+1.5),alpha=.5,label=exp)
ax.axvline(0,color='black',linestyle='--');ax.set(xlabel='Per-scene TP difference (agreement − confidence)',ylabel='Number of scenes',title='Impact of agreement selection');ax.legend()
plt.tight_layout();plt.savefig(OUT/'per_image_tp_delta.png',dpi=170);plt.show()


## 6. Paired scene-level bootstrap (exploratory uncertainty)
Resample scenes, not individual boxes, because predictions within a scene are correlated. 95% percentile interval is descriptive; small samples and selection of scenes limit generalization. The 20-scene photometric set is **not independent** of the 50-scene flip set.

In [ ]:
rng=np.random.default_rng(42)
boot_rows=[]
for exp,g in image_df.groupby('experiment'):
    wide=g.pivot(index='image',columns='method',values=['TP','retained'])
    tpc=wide[('TP','confidence_only')].to_numpy(dtype=float)
    tpa=wide[('TP','confidence_agreement')].to_numpy(dtype=float)
    k=wide[('retained','confidence_only')].to_numpy(dtype=float)
    observed=(tpa.sum()-tpc.sum())/k.sum()*100
    idx=rng.integers(0,len(k),size=(5000,len(k)))
    delta=((tpa[idx].sum(axis=1)-tpc[idx].sum(axis=1))/k[idx].sum(axis=1))*100
    lo,hi=np.percentile(delta,[2.5,97.5])
    boot_rows.append({'experiment':exp,'n_scenes':len(k),'precision_delta_pp':observed,
                      'bootstrap_low_pp':lo,'bootstrap_high_pp':hi})
bootstrap=pd.DataFrame(boot_rows)
bootstrap.to_csv(OUT/'paired_scene_bootstrap.csv',index=False)
display(bootstrap)


## 7. Export guide-ready evidence report and manifest
This is an **offline diagnostic report**, not a claim of method improvement or completed training comparisons.

In [ ]:
def table_text(df):
    try:return df.to_markdown(index=False,floatfmt='.4f')
    except ImportError:return df.to_string(index=False)

f50=agg[agg.experiment=='original_flip_50'].copy()
p20=agg[agg.experiment=='independent_photometric_20'].copy()
low=band[band.band=='0.70–0.80'].copy()
manifest={k:{'path':str(p),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for k,p in FILES.items()}
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
report=f"""# Experiment 10 — Fixed-teacher pseudo-label agreement: failure audit

Generated: {datetime.datetime.now().isoformat(timespec='seconds')}

## Question and protocol
Frozen teacher on strong-fog validation images. Original-vs-flip 50 scenes; independent weak photometric views 20 scenes. Per-image matched K fixed **before GT evaluation**. Class-aware greedy IoU ≥ 0.5 matching. Crowd annotations excluded. These metrics are diagnostic P/R/F1, not mAP.

## 50-scene original/flip comparison
{table_text(f50)}

## 20-scene independent photometric comparison
{table_text(p20)}

## 0.70–0.80 selected-box evidence
{table_text(low)}

## Scene-level uncertainty (exploratory)
{table_text(bootstrap)}

## Selection swaps
{table_text(swap_df.groupby('experiment',as_index=False)[['K','shared','swapped_pairs']].sum())}

## Interpretation and limitations
- The existing 50-scene comparison is a **negative result** for the tested confidence×agreement ranking; no improvement is claimed.
- The 0.70–0.80 results describe **selected boxes**, not all candidates in that band. Confidence-only and agreement-selected subsets differ, and per-set GT matching is set-dependent.
- Ground truth was consulted only **after** the original selection files were produced; this notebook evaluates those saved selections.
- These are small, nonrandom scene subsets, not repeated-seed training studies. No statistical significance or generalization claim follows from bootstrap intervals alone.
- Crowd ground truth is excluded; this is not full COCO crowd-ignore evaluation.
- No matched downstream training, fog-level ablation, or second-dataset validation is performed here.
- Historical training logs verify recorded configuration and execution but not exact historical target sample identities.

## Next experiment (not yet completed)
Inspect the complete View-A candidate pool, particularly 0.70–0.80 scores, for stable **true positives versus stable false positives**. Define a refinement **without using validation GT for training-time selection**, then run equal-budget baseline vs refined training with matched seed/config, multiple seeds and another fog dataset. Report target mAP@0.5 and full COCO AP as appropriate.

## Harmonious Teacher (CVPR 2023)
Related prior work addresses pseudo-label classification/localization harmony. This experiment is frozen-teacher, two-view class/box agreement with matched-count ranking; it is **not** a Harmonious Teacher reproduction. Novelty and comparisons must be verified against the paper.

## Artifact provenance
See `input_manifest.json` for SHA-256 of the three source files. Output CSV and PNG files in this folder are derived diagnostic evidence.
"""
(OUT/'GUIDE_READY_FAILURE_ANALYSIS.md').write_text(report,encoding='utf-8')
print('SAVED:',OUT/'GUIDE_READY_FAILURE_ANALYSIS.md')
print('Files:')
for p in sorted(OUT.iterdir()):print(' ',p.name)
print('\nCONCLUSION: Offline failure analysis completed. Downstream training improvement NOT established.')
